# Async Demo Manual Mode
This notebook runs the async pipeline modules step by step while manually providing the rate data artifact.

In [ ]:
from pathlib import Path

from simkit.config import defaults, schema
from simkit.core.pipeline import entry_point_validate
from simkit.io import readers, writers

from battery_tea import schemas
from battery_tea.io import read_parquet_load_profile
from battery_tea.modules.battery_config import ConfigureBatteryModule
from battery_tea.modules.cost_calc import CostCalculatorModule
from battery_tea.modules.perf_sim_simple import SimplePerformanceSimModule
from battery_tea.modules.project_analyzer import ProjectAnalyzerModule


In [ ]:
# This notebook assumes it is run from the packages/battery-tea-demo directory
spec_path = Path("battery_tea/tests/fixtures/pipeline_configs/demo_linear_alt.yaml")
specification = entry_point_validate(spec_path)
entry_spec = specification.modules["entry_point"]
base_dir = specification.source_path.parent if specification.source_path else spec_path.parent

fixture_dir = spec_path.parent.parent

def _resolve_entry_artifact(channel: str) -> Path:
    binding = entry_spec.outputs[channel]
    artifact_path = binding.artifact_path
    if artifact_path is None:
        raise ValueError(f"Entry channel {channel!r} is missing an artifact path")
    return (base_dir / artifact_path).resolve()

geography = readers.read_json_model(_resolve_entry_artifact("geo"), schemas.Geography)
load_profile_path = _resolve_entry_artifact("load_profile")
rate_info_manual = readers.read_json_model(
    fixture_dir / "rateinfo_tou_synthetic.json", schemas.RateInfo
).model_copy(update={"source": "manual_notebook_override"})
assert len(rate_info_manual.energy_price_usd_per_kwh) == 8760


In [ ]:
load_profile = read_parquet_load_profile(load_profile_path, source="manual_notebook")
battery_module = ConfigureBatteryModule()
battery_config = battery_module.run(load_profile, rate_info_manual, None).data
cost_module = CostCalculatorModule()
cost_breakdown = cost_module.run(battery_config, geography).data
perf_module = SimplePerformanceSimModule()
telemetry = perf_module.run(battery_config, load_profile, None, rate_info_manual).data
analyzer_module = ProjectAnalyzerModule()
financial_results = analyzer_module.run(
    rate_info_manual, telemetry, defaults.default_financial_params(), cost_breakdown
).data
for series in (telemetry.charge_in_kwh, telemetry.discharge_out_kwh, telemetry.soc_kwh):
    assert len(series) == 8760


In [ ]:
# Summary of results
results = {
    "battery_capacity_kwh": battery_config.capacity_kwh,
    "battery_power_kw": battery_config.power_kw,
    "capex_total": cost_breakdown.capex_total,
    "annual_om_usd": cost_breakdown.annual_om_usd,
    "annual_savings": financial_results.annual_savings,
    "npv": financial_results.npv,
    "irr": financial_results.irr,
    "payback_years": financial_results.payback_years,
}
print("Manual mode results:")
for key, value in results.items():
    print(f"  {key}: {value}")

# Optionally save outputs to disk
output_dir = Path("notebooks/outputs/manual_mode")
output_dir.mkdir(parents=True, exist_ok=True)
writers.write_json_model(rate_info_manual, output_dir / "rate_info_manual.json")
writers.write_json_model(battery_config, output_dir / "battery_config.json")
writers.write_json_model(cost_breakdown, output_dir / "cost_breakdown.json")
writers.write_json_model(financial_results, output_dir / "financial_results.json")
print(f"\nOutputs saved to: {output_dir}")
